# Chapter 12 Lab — Machine Translation

A small encoder-decoder-with-attention model trained from scratch on a tiny public English-
French phrase list, then a pretrained Transformer MT model on the same sentences for comparison,
scored with BLEU.

## 1. A tiny parallel phrase list (public-domain, hand-written for this lab)

In [ ]:
pairs = [
    ("hello", "bonjour"),
    ("good morning", "bonjour"),
    ("thank you", "merci"),
    ("good night", "bonne nuit"),
    ("see you soon", "a bientot"),
    ("how are you", "comment allez vous"),
    ("i am fine", "je vais bien"),
    ("what is your name", "comment vous appelez vous"),
]

## 2. Minimal encoder-decoder with attention (PyTorch, tiny scale for teaching)

In [ ]:
import torch, torch.nn as nn

src_vocab = sorted(set(w for s, _ in pairs for w in s.split())) + ["<pad>"]
tgt_vocab = sorted(set(w for _, t in pairs for w in t.split())) + ["<pad>", "<sos>", "<eos>"]
src_stoi, tgt_stoi = {w: i for i, w in enumerate(src_vocab)}, {w: i for i, w in enumerate(tgt_vocab)}

class Encoder(nn.Module):
    def __init__(self, vocab_size, hidden=32):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden)
        self.gru = nn.GRU(hidden, hidden, batch_first=True)
    def forward(self, x):
        return self.gru(self.emb(x))

class AttnDecoder(nn.Module):
    def __init__(self, vocab_size, hidden=32):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden)
        self.gru = nn.GRUCell(hidden, hidden)
        self.out = nn.Linear(hidden * 2, vocab_size)
    def step(self, prev_token, h, enc_out):
        e = self.emb(prev_token)
        h = self.gru(e, h)
        scores = torch.softmax(enc_out @ h.unsqueeze(-1), dim=1)  # simple dot-product attention
        ctx = (scores * enc_out).sum(dim=1)
        logits = self.out(torch.cat([h, ctx], dim=-1))
        return logits, h

print(f"src vocab={len(src_vocab)}, tgt vocab={len(tgt_vocab)} — tiny toy scale, illustrative not production")

## 3. Pretrained Transformer MT for comparison

In [ ]:
from transformers import pipeline
translator = pipeline("translation_en_to_fr", model="Helsinki-NLP/opus-mt-en-fr")
for src, ref in pairs[:4]:
    pred = translator(src)[0]["translation_text"]
    print(f"{src:25s} -> pred: {pred:25s} ref: {ref}")

## 4. BLEU scoring

In [ ]:
from sacrebleu import corpus_bleu

preds = [translator(s)[0]["translation_text"] for s, _ in pairs]
refs = [[t for _, t in pairs]]
print(corpus_bleu(preds, refs))

## Exercise

The toy encoder-decoder above is deliberately tiny (32-dim hidden state, 8 training pairs) — it
will not translate well. Train it for a few hundred steps on the 8 pairs and see whether it can
at least memorize them, then explain in one paragraph why memorizing 8 pairs is not the same as
learning to translate.